# Emit Vitis-synthesis LLVM IR for an hls4ml project

The Vitis HLS frontend is `clang-3.9-csynth`, not `clang-16`. It requires `-fhls`, the FPGA target, and `autopilot_ssdm_op.h`; these make `ssdm_int` and `__SYNTHESIS__` work without changing the project's open-source AP headers.

The notebook first preprocesses raw `firmware/myproject.cpp` with the installed HLS Clang; it does not require a project-local `.autopilot` directory or a prior HLS run. The final cleanup only removes Vitis-specific `!fpga.*` metadata attachments so LLVM 16 / ProGraML can parse the otherwise valid, legacy LLVM IR.

In [12]:
from pathlib import Path
import os
import re
import subprocess
import time

project_dir = Path('/home/brend/projects/data/cache/extracted/2layer/archive_4/projects/dense_8_64_88_10b_rf3070')
source = project_dir / 'firmware/myproject.cpp'
output_dir = Path.cwd() / 'vitis_llvm_ir'
output_dir.mkdir(exist_ok=True)
preprocessed_source = output_dir / 'myproject-vitis.pp.cpp'
output_bc = output_dir / 'myproject-vitis.bc'
output_ll = output_dir / 'myproject-vitis.ll'
graph_path = output_dir / 'myproject-vitis.json'
pragma_dump_path = output_dir / 'myproject-vitis.pragmas.log'

vitis = Path('/opt/Xilinx/2025.2/Vitis')
clang = Path('/opt/Xilinx/2025.2/lnx64/tools/clang-3.9-csynth/bin/clang')
llvm_dis = Path('/usr/bin/llvm-dis-16')
llvm_as = Path('/usr/bin/llvm-as-16')
assert source.is_file()
assert all(path.is_file() for path in (clang, llvm_dis, llvm_as))

env = os.environ.copy()
vitis_lib = vitis / 'lib/lnx64.o'
env['LD_LIBRARY_PATH'] = ':'.join(filter(None, [str(vitis_lib), env.get('LD_LIBRARY_PATH')]))
print(subprocess.run([str(clang), '--version'], env=env, check=True, text=True, capture_output=True).stdout)

clang version 7.0.0 
Target: x86_64-unknown-linux-gnu
Thread model: posix
InstalledDir: /opt/Xilinx/2025.2/lnx64/tools/clang-3.9-csynth/bin



In [13]:
autopilot = vitis / 'common/technology/autopilot'
autopilot_sysc = autopilot / 'ap_sysc'
ap_types = project_dir / 'firmware/ap_types'
common_args = [
    '-fhls', '-fno-exceptions', '-fno-math-errno', '-fno-threadsafe-statics',
    '-fno-use-cxa-atexit', '-target', 'fpga64-xilinx-linux-gnu',
    '-D__VITIS_HLS__', '-DAESL_SYN', '-D__SYNTHESIS__', '-D__HLS_SYN__',
    f'-I{autopilot}', f'-I{autopilot_sysc}', f'-I{ap_types}',
    '-include', str(autopilot / 'etc/autopilot_ssdm_op.h'),
]
preprocess_cmd = [str(clang), str(source), '-E', '-std=c++0x', *common_args, '-o', str(preprocessed_source)]
subprocess.run(preprocess_cmd, cwd=project_dir, env=env, check=True, timeout=120)

cmd = [
    str(clang), str(preprocessed_source), '-c', '-emit-llvm', '-flto',
    '-fno-exceptions', '-fno-math-errno', '-fno-threadsafe-statics', '-fno-use-cxa-atexit',
    '-Wpragmas', '-Wdump-hls-pragmas', '-Wno-error=dump-hls-pragmas', *common_args,
    '-o', str(output_bc),
]
start = time.perf_counter()
result = subprocess.run(cmd, cwd=project_dir, env=env, text=True, capture_output=True, timeout=120)
pragma_dump_path.write_text(result.stderr)
print(result.stderr[-4000:])  # pragma-dump warnings are expected
result.check_returncode()
print(f'Wrote {output_bc} in {time.perf_counter() - start:.1f}s')

/dense_8_64_88_10b_rf3070/firmware/nnet_utils/nnet_dense_stream.h:64:9: warning: HLS pragma dump _XLX_SEP_ PragmaIsValid=1_XLX_SEP_ PragmaType=unroll_XLX_SEP_ PragmaContext=_XLX_SEP_ PragmaFunction=res_write_XLX_SEP_ PragmaOptions=_XLX_SEP_ [-Wdump-hls-pragmas]
#pragma HLS UNROLL
        ^
/home/brend/projects/data/cache/extracted/2layer/archive_4/projects/dense_8_64_88_10b_rf3070/firmware/nnet_utils/nnet_dense_stream.h:74:9: warning: HLS pragma dump _XLX_SEP_ PragmaIsValid=1_XLX_SEP_ PragmaType=unroll_XLX_SEP_ PragmaContext=_XLX_SEP_ PragmaFunction=res_write_XLX_SEP_ PragmaOptions=_XLX_SEP_ [-Wdump-hls-pragmas]
#pragma HLS UNROLL
        ^
/home/brend/projects/data/cache/extracted/2layer/archive_4/projects/dense_8_64_88_10b_rf3070/firmware/nnet_utils/nnet_dense_stream.h:85:9: warning: HLS pragma dump _XLX_SEP_ PragmaIsValid=1_XLX_SEP_ PragmaType=inline_XLX_SEP_ PragmaContext=_XLX_SEP_ PragmaFunction=dense_XLX_SEP_ PragmaOptions=recursive_XLX_SEP_ [-Wdump-hls-pragmas]
#pragma HLS INLIN

In [17]:
# Upgrade LLVM 7 bitcode to LLVM 16 textual IR. Vitis's !fpga.* attachments
# use metadata shapes no longer accepted by LLVM 16, so remove those attachments
# before reassembling. This does not change instructions or ordinary LLVM metadata.
legacy_ll = output_dir / 'myproject-vitis-legacy.ll'
clean_ll = output_dir / 'myproject-vitis-clean.ll'
clean_bc = output_dir / 'myproject-vitis-clean.bc'
subprocess.run([str(llvm_dis), '-opaque-pointers=0', str(output_bc), '-o', str(legacy_ll)], check=True)
clean_text = re.sub(r' !fpga\.[A-Za-z0-9_.]+ !\d+', '', legacy_ll.read_text())
clean_ll.write_text(clean_text)
subprocess.run([str(llvm_as), '-opaque-pointers=0', str(clean_ll), '-o', str(clean_bc)], check=True)
subprocess.run([str(llvm_dis), '-opaque-pointers=0', str(clean_bc), '-o', str(output_ll)], check=True)
assert 'declare void @fprintf' not in output_ll.read_text()
print(f'Wrote {output_ll} ({output_ll.stat().st_size / 2**20:.1f} MiB)')

Wrote /home/brend/projects/ll-hls4ml/notebooks/vitis_llvm_ir/myproject-vitis.ll (2.0 MiB)


DILocation not allowed within this metadata node
!6932 = !{!"fpga.dataflow.func", !"user", !6933}
!6933 = !DILocation(line: 15, column: 9, scope: !6923)
DILocation not allowed within this metadata node
!6965 = !{!"fpga.inline", !"user", !6966}
!6966 = !DILocation(line: 39, column: 9, scope: !6953)
DILocation not allowed within this metadata node
!6992 = !{!"fpga.inline", !"user", !6993}
!6993 = !DILocation(line: 39, column: 9, scope: !6983)
DILocation not allowed within this metadata node
!7015 = !{!"fpga.inline", !"user", !7016}
!7016 = !DILocation(line: 720, column: 23, scope: !7010)
DILocation not allowed within this metadata node
!7047 = !{!"fpga.inline", !"user", !7048}
!7048 = !DILocation(line: 522, column: 39, scope: !7032)
DILocation not allowed within this metadata node
!7072 = !{!"fpga.inline", !"user", !7073}
!7073 = !DILocation(line: 522, column: 39, scope: !7057)
DILocation not allowed within this metadata node
!7097 = !{!"fpga.inline", !"user", !7098}
!7098 = !DILocation(

In [15]:
llvm2graph = Path('/home/brend/projects/ir_parsing/ProGraML/bazel-bin/programl/bin/llvm2graph-16')
graph2json = Path('/home/brend/projects/ir_parsing/ProGraML/bazel-bin/programl/bin/graph2json')
assert llvm2graph.is_file() and graph2json.is_file()

start = time.perf_counter()
ir_result = subprocess.run([str(llvm2graph), str(output_ll)], capture_output=True, timeout=1000)
if ir_result.returncode:
    raise RuntimeError(ir_result.stderr.decode(errors='replace')[:1000])
json_result = subprocess.run([str(graph2json)], input=ir_result.stdout, capture_output=True, timeout=1000)
if json_result.returncode:
    raise RuntimeError(json_result.stderr.decode(errors='replace')[:1000])
graph_path.write_bytes(json_result.stdout)
from ll_hls4ml.pragmas import inject_vitis_pragmas
pragma_stats = inject_vitis_pragmas(graph_path, pragma_dump_path)
print(pragma_stats)
print(f'Wrote {graph_path} ({graph_path.stat().st_size / 2**20:.1f} MiB) in {time.perf_counter() - start:.1f}s')

{'pragma_dump_records': 66, 'carrier_pragmas_injected': 7, 'dump_pragmas_injected': 26, 'pragma_nodes_injected': 33, 'pragmas_unmatched': 27}
Wrote /home/brend/projects/ll-hls4ml/notebooks/vitis_llvm_ir/myproject-vitis.json (7.4 MiB) in 3.6s
